# Fantasy Premier League Data Pipeline
This notebook replicates the functionality of `pipeline.py` for processing Fantasy Premier League data. It loads, cleans, enriches, and saves the dataset for further analysis or modeling.

## 1. Import Required Libraries
Import all necessary libraries, including os, pandas, numpy, and unidecode.

In [1]:
import os
import pandas as pd
import numpy as np
import uuid as uuid_module 
import re
from unidecode import unidecode

In [2]:
import os

# Check current working directory
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")



Current directory: c:\Python\fpl_pipeline\vaastav_dataset
Contents: ['check_empty_ut.py', 'check_name_duplicates.py', 'download_data.py', 'fix_encoding.py', 'fpl_pipeline_notebook.ipynb', 'output']


## 2. Set Data Paths
Define variables for `DATA_ROOT` and `OUTPUT_FILE` to specify input and output file locations.

In [3]:
# === Canonical paths (relative to the notebook INSIDE vaastav_dataset/) ===
DATA_ROOT = "../data"              # root/data
OUTPUT_DIR = "../output"           # root/output
OUTPUT_FILE = f"{OUTPUT_DIR}/training_data.csv"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Output file: {OUTPUT_FILE}")

Data root: ../data
Output dir: ../output
Output file: ../output/training_data.csv


## 3. Normalize Player Names
Define and use a function to normalize player names using unidecode and string operations.


In [4]:

def normalize_player_name(name):
    """
    Normalizes player names by converting to lowercase, removing accents, 
    and critically, stripping the trailing numeric IDs (e.g., " 534").
    """
    if pd.isnull(name):
        return name
    
    # Convert to string and lowercase
    name = str(name).strip().lower()
    
    # Remove accents
    name = unidecode(name)
    
    # Remove trailing numbers with or without spaces
    # This handles: "aaron connolly 534" and "aaron connolly534"
    name = re.sub(r'\s*\d+\s*$', '', name)
    
    # Clean up underscores
    name = name.replace("_", " ")
    
    # Keep only alphanumeric and spaces
    name = "".join(c for c in name if c.isalnum() or c.isspace())
    
    # Remove multiple spaces
    name = " ".join(name.split())
    
    return name

In [5]:

# CELL 3: Test Normalization Function

print("\n=== TESTING NORMALIZATION FUNCTION ===")
test_names = ["aaron cresswell 376", "aaron cresswell 402", "Aaron Connolly 534", "Son Heung-Min 123"]
for test in test_names:
    result = normalize_player_name(test)
    print(f"  '{test}' -> '{result}'")
print("======================================\n")

test_names = ["aaron cresswell 376", "aaron connolly 534", "Aaron Connolly 534", "Son Heung-Min 123"]
for test in test_names:
    result = normalize_player_name(test)
    print(f"  '{test}' -> '{result}'")


=== TESTING NORMALIZATION FUNCTION ===
  'aaron cresswell 376' -> 'aaron cresswell'
  'aaron cresswell 402' -> 'aaron cresswell'
  'Aaron Connolly 534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'

  'aaron cresswell 376' -> 'aaron cresswell'
  'aaron connolly 534' -> 'aaron connolly'
  'Aaron Connolly 534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'


## 4. Load All Gameweeks Data
Implement and run the function to load and concatenate all gameweek CSV files from the data directory.

In [6]:
def load_all_gameweeks(data_root=DATA_ROOT):
    """Load and concatenate all gameweek CSVs."""
    all_data = []
    print("\nLoading Gameweek Data...")
    
    for season in sorted(os.listdir(data_root)):
        season_path = os.path.join(data_root, season)

        # 👇 Skip non-folders or folders we handle separately
        if not os.path.isdir(season_path):
            continue
        if season == "2025-26":
            print(f"⏩ Skipping season {season} (handled separately via merged_dataset.csv)")
            continue

        # --- Determine where to look for GW files ---
        gws_path = os.path.join(season_path, "gws")
        search_path = gws_path if os.path.exists(gws_path) else season_path

        try:
            all_files = os.listdir(search_path)
        except Exception as e:
            print(f"  ERROR accessing {search_path}: {e}")
            continue

        # --- Select only valid gameweek files (gw1.csv, gw2.csv, etc.) ---
        gw_files = sorted(
            [
                f for f in all_files
                if f.startswith("gw")
                and f.endswith(".csv")
                and f[2:-4].isdigit()  # only filenames like gw1.csv, gw2.csv, etc.
            ],
            key=lambda x: int(x.replace("gw", "").replace(".csv", ""))
        )

        if not gw_files:
            print(f"  Season: {season} - No gameweek files found")
            continue

        print(f"  Season: {season} ({len(gw_files)} gameweeks)")

        # --- Read each GW file ---
        for filename in gw_files:
            gw_number = int(filename.replace("gw", "").replace(".csv", ""))
            gw_path = os.path.join(search_path, filename)

            try:
                df = pd.read_csv(gw_path, encoding="utf-8")
                df["season"] = season
                df["Gameweek"] = gw_number
                all_data.append(df)
            except Exception as e:
                print(f"    ERROR reading {filename}: {e}")

    # --- Combine all seasons ---
    if all_data:
        combined = pd.concat(all_data, ignore_index=True)
        print(f"\n✅ Total gameweek records loaded: {len(combined):,}")
        return combined
    else:
        print("⚠️ No gameweek data found.")
        return pd.DataFrame()



In [7]:
# Load all gameweek data
df_gws = load_all_gameweeks()




Loading Gameweek Data...
  Season: 2018-19 (38 gameweeks)
  Season: 2019-20 (38 gameweeks)
  Season: 2020-21 (38 gameweeks)
  Season: 2021-22 (38 gameweeks)
  Season: 2022-23 (38 gameweeks)
  Season: 2023-24 (38 gameweeks)
  Season: 2024-25 (38 gameweeks)
⏩ Skipping season 2025-26 (handled separately via merged_dataset.csv)

✅ Total gameweek records loaded: 171,993


C:\Users\SOFI\AppData\Local\Temp\ipykernel_13796\2338496332.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_data, ignore_index=True)


In [8]:
import os
import pandas as pd

def load_merged_season_2025_26(data_root=DATA_ROOT):
    """
    Loads the merged_dataset.csv for the 2025–26 season
    and reformats it to match the structure of the older seasons.
    """
    season_folder = os.path.join(data_root, "2025-26")
    merged_path = os.path.join(season_folder, "merged_dataset.csv")

    if not os.path.exists(merged_path):
        print("⚠️  2025–26 merged_dataset.csv not found — skipping this season.")
        return pd.DataFrame()

    print(f"\n📥 Loading merged dataset for 2025–26 from {merged_path}")
    df_new = pd.read_csv(merged_path)

    rename_map = {
        "gw": "Gameweek",
        "player_id": "element",
        "team_name": "team",            # ✅ Εδώ ήταν το πρόβλημα!
        "web_name": "web_name",
        "position": "position",
        "minutes": "minutes",
        "goals_scored": "goals_scored",
        "assists": "assists",
        "total_points": "total_points",
        "goals_conceded": "goals_conceded",
        "clean_sheets": "clean_sheets",
        "yellow_cards": "yellow_cards",
        "red_cards": "red_cards",
        "bonus": "bonus",
        "influence": "influence",
        "creativity": "creativity",
        "threat": "threat",
        "ict_index": "ict_index",
        "first_name": "first_name",
        "second_name": "second_name"
    }
    df_new.rename(columns={k: v for k, v in rename_map.items() if k in df_new.columns}, inplace=True)

    # --- Add season label ---
    df_new["season"] = "2025-26"

    # --- Ensure critical columns exist ---
    expected_cols = [
        "season", "Gameweek", "element", "web_name", "team", "position",
        "minutes", "goals_scored", "assists", "total_points"
    ]
    for col in expected_cols:
        if col not in df_new.columns:
            df_new[col] = pd.NA

    print(f"✅ Loaded 2025–26 dataset: {df_new.shape[0]:,} rows, {df_new.shape[1]:,} columns")
    return df_new


# %%
# Load and append 2025–26 season to df_gws
df_2025 = load_merged_season_2025_26()

if not df_2025.empty:
    df_gws = pd.concat([df_gws, df_2025], ignore_index=True)
    print(f"✅ Combined dataset now has {len(df_gws):,} total rows (including 2025–26)")
else:
    print("⚠️  No 2025–26 data appended.")


📥 Loading merged dataset for 2025–26 from ../data\2025-26\merged_dataset.csv
✅ Loaded 2025–26 dataset: 7,311 rows, 36 columns
✅ Combined dataset now has 179,304 total rows (including 2025–26)


In [9]:
print("\n✅ 2025–26 quick check:")
if 'season' in df_2025.columns:
    print(df_2025['season'].unique())
print("Rows:", len(df_2025))
display(df_2025.head(3))



✅ 2025–26 quick check:
['2025-26']
Rows: 7311


,first_name,second_name,web_name,team,position,Gameweek,element,minutes,goals_scored,assists,...,tackles,defensive_contribution,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,in_dreamteam,season
0,David,Raya Martín,Raya,Arsenal,GKP,1,1,90,0,0,...,0,0,1,0.0,0.0,0.0,1.52,10,True,2025-26
1,Kepa,Arrizabalaga Revuelta,Arrizabalaga,Arsenal,GKP,1,2,0,0,0,...,0,0,0,0.0,0.0,0.0,0.00,0,False,2025-26
2,Karl,Hein,Hein,Arsenal,GKP,1,3,0,0,0,...,0,0,0,0.0,0.0,0.0,0.00,0,False,2025-26


In [10]:
# Preview Gameweeks Data
print("\nPreview of raw gameweek data:")
display(df_gws.head())
print(f"\nColumns: {list(df_gws.columns)}")
print(f"Shape: {df_gws.shape}")


Preview of raw gameweek data:


,name,assists,attempted_passes,big_chances_created,big_chances_missed,bonus,bps,clean_sheets,clearances_blocks_interceptions,completed_passes,...,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,first_name,second_name,web_name,defensive_contribution,in_dreamteam
0,Aaron_Cresswell_402,0,0.0,0.0,0.0,0,0,0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aaron_Lennon_83,0,22.0,0.0,1.0,0,6,1,1.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aaron_Mooy_199,0,51.0,0.0,0.0,0,24,0,2.0,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aaron_Ramsey_14,0,11.0,0.0,0.0,0,7,0,0.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aaron_Wan-Bissaka_145,1,29.0,1.0,0.0,3,38,1,11.0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Columns: ['name', 'assists', 'attempted_passes', 'big_chances_created', 'big_chances_missed', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'completed_passes', 'creativity', 'dribbles', 'ea_index', 'element', 'errors_leading_to_goal', 'errors_leading_to_goal_attempt', 'fixture', 'fouls', 'goals_conceded', 'goals_scored', 'ict_index', 'id', 'influence', 'key_passes', 'kickoff_time', 'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'minutes', 'offside', 'open_play_crosses', 'opponent_team', 'own_goals', 'penalties_conceded', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackled', 'tackles', 'target_missed', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'winning_goals', 'yellow_cards', 'season', 'Gameweek', 'position', 'team', 'xP', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', '

In [11]:
# %%
# NEW CELL: Load Player Web Names

def load_player_web_names(data_root=DATA_ROOT):
    """Load player ID, full name, and web name for each season."""
    player_data_by_season = {}
    print("\nLoading Player Web Names...")
    
    for season in sorted(os.listdir(data_root)):
        season_path = os.path.join(data_root, season)
        if not os.path.isdir(season_path):
            continue
            
        # The file containing web_name is usually 'players_raw.csv' or similar in the season folder
        player_list_file = os.path.join(season_path, "players_raw.csv")
        if not os.path.exists(player_list_file):
            # Fallback for older seasons or different structures
            player_list_file = os.path.join(season_path, "player_idlist.csv")
            if not os.path.exists(player_list_file):
                 continue

        try:
            # We only need the ID/element, name, and web_name
            p_df = pd.read_csv(player_list_file, encoding="utf-8")
            
            # Standardizing column names for merging
            # Standardize the player identifier to "element" everywhere
            if 'element' not in p_df.columns:
                if 'id' in p_df.columns:
                    p_df.rename(columns={'id': 'element'}, inplace=True)
                elif 'Code' in p_df.columns:
                     p_df.rename(columns={'Code': 'element'}, inplace=True)

             # Keep only element and web_name
            if 'element' in p_df.columns and 'web_name' in p_df.columns:
                p_df = p_df[['element', 'web_name']].copy()
                p_df['element'] = pd.to_numeric(p_df['element'], errors='coerce').astype('Int64')
                player_data_by_season[season] = p_df.set_index('element')
                print(f"  Season: {season} - Loaded {len(p_df)} player names.")
            else:
                print(f"  Season: {season} - Missing 'element' or 'web_name' columns.")

                print(f"  Season: {season} - Loaded {len(p_df)} player names.")
            

        except Exception as e:
            print(f"  ERROR loading player list for {season}: {e}")
            
    if player_data_by_season:
        print(f" → Loaded player web names for {len(player_data_by_season)} seasons.")
    return player_data_by_season

# Load player web names
player_web_names_by_season = load_player_web_names()

print("\nLoaded Web Names Preview for a random season:")
if player_web_names_by_season:
    first_season = next(iter(player_web_names_by_season.values()))
    display(first_season.head())
# %% [markdown]
# ## 4B. Merge Web Names Data
# Merge the loaded player web names into the main gameweek data.
# %%
# Merge web names into the main dataframe
def merge_web_names(df_gws, player_web_names_by_season):
    """Merge web_name from separate data files into the main gameweek DataFrame."""
    print("\nMerging Web Names into Gameweek Data...")

    # Ενοποίηση τύπου στη στήλη 'element'
    df_gws['element'] = pd.to_numeric(df_gws['element'], errors='coerce').astype('Int64')
   
    # 🧩 Skip 2025–26 because merged_dataset already has correct web_name values
    if '2025-26' in df_gws['season'].unique():
        mask = df_gws['season'] == '2025-26'
        count = df_gws.loc[mask, 'web_name'].notna().sum()
        print(f"⏩ Skipping merge for 2025–26 (already has {count} web_name values)")
        player_web_names_by_season = {k: v for k, v in player_web_names_by_season.items() if k != '2025-26'}
    
    if 'web_name' not in df_gws.columns:
        df_gws['web_name'] = pd.NA


    # Merge ανά season
    for season, p_df_map in player_web_names_by_season.items():
        mask = df_gws['season'] == season
        if not mask.any():
            continue

        df_season = df_gws.loc[mask].copy()

        # Εκτέλεση merge
        merged = df_season.merge(
            p_df_map[['web_name']],
            left_on='element',
            right_index=True,
            how='left'
        )

        # Αν το pandas πρόσθεσε suffix (_x, _y), τα χειριζόμαστε
        if 'web_name_y' in merged.columns:
            merged['web_name'] = merged['web_name_y']
        elif 'web_name_x' in merged.columns:
            merged['web_name'] = merged['web_name_x']

        df_gws.loc[mask, 'web_name'] = merged['web_name'].values

    df_gws['web_name'] = df_gws['web_name'].replace(['nan', ''], np.nan)
    print(f"✅ Web names filled: {df_gws['web_name'].notna().sum():,}/{len(df_gws):,} rows")

    return df_gws



Loading Player Web Names...
  Season: 2018-19 - Loaded 624 player names.
  Season: 2019-20 - Loaded 666 player names.
  Season: 2020-21 - Loaded 713 player names.
  Season: 2021-22 - Loaded 737 player names.
  Season: 2022-23 - Loaded 778 player names.
  Season: 2023-24 - Loaded 865 player names.
  Season: 2024-25 - Loaded 804 player names.
 → Loaded player web names for 7 seasons.

Loaded Web Names Preview for a random season:


,web_name
element,
1,Cech
2,Leno
3,Koscielny
4,Bellerín
5,Monreal


In [12]:
# Run the merge
df_gws = merge_web_names(df_gws, player_web_names_by_season)

print("\nGameweek Data Preview with new 'web_name' column:")
display(df_gws[['season', 'Gameweek', 'name', 'web_name', 'element']].head())



Merging Web Names into Gameweek Data...
⏩ Skipping merge for 2025–26 (already has 7311 web_name values)
✅ Web names filled: 179,304/179,304 rows

Gameweek Data Preview with new 'web_name' column:


,season,Gameweek,name,web_name,element
0,2018-19,1,Aaron_Cresswell_402,Cresswell,402
1,2018-19,1,Aaron_Lennon_83,Lennon,83
2,2018-19,1,Aaron_Mooy_199,Mooy,199
3,2018-19,1,Aaron_Ramsey_14,Ramsey,14
4,2018-19,1,Aaron_Wan-Bissaka_145,Wan-Bissaka,145


## 5. Load Fixtures Data
Implement and run the function to load and concatenate all fixtures CSV files from the data directory.

In [13]:
def load_fixtures(data_root=DATA_ROOT):
    """Load and concatenate all fixtures CSVs."""
    all_fx = []
    print("\nLoading Fixtures Data...")
    for season in sorted(os.listdir(data_root)):
        season_path = os.path.join(data_root, season)
        if not os.path.isdir(season_path):
            continue
        fixtures_file = os.path.join(season_path, "fixtures.csv")
        if not os.path.exists(fixtures_file):
            continue
        try:
            fx = pd.read_csv(fixtures_file, encoding="utf-8")
            fx["season"] = season
            all_fx.append(fx)
        except Exception as e:
            print(f"  ERROR loading fixtures for {season}: {e}")
    if all_fx:
        print(f"Loaded fixtures for {len(all_fx)} seasons.")
        return pd.concat(all_fx, ignore_index=True)
    else:
        print("No fixtures data found.")
        return pd.DataFrame()

In [14]:
# Load fixtures data
fixtures = load_fixtures()
# Preview Fixtures Data
print("\nPreview of fixtures data:")
display(fixtures.head())
print(f"Shape: {fixtures.shape}")



Loading Fixtures Data...
Loaded fixtures for 8 seasons.

Preview of fixtures data:


,code,deadline_time,deadline_time_formatted,event,event_day,finished,finished_provisional,id,kickoff_time,kickoff_time_formatted,...,started,stats,team_a,team_a_difficulty,team_a_score,team_h,team_h_difficulty,team_h_score,season,pulse_id
0,987597.0,2018-08-10T18:00:00Z,10 Aug 19:00,1,1.0,True,True,6,2018-08-10T19:00:00Z,10 Aug 20:00,...,True,"[{'goals_scored': {'a': [{'value': 1, 'element...",11,4,1.0,14,3,2.0,2018-19,NaN
1,987598.0,2018-08-10T18:00:00Z,10 Aug 19:00,1,2.0,True,True,7,2018-08-11T11:30:00Z,11 Aug 12:30,...,True,"[{'goals_scored': {'a': [{'value': 1, 'element...",17,3,2.0,15,4,1.0,2018-19,NaN
2,987592.0,2018-08-10T18:00:00Z,10 Aug 19:00,1,2.0,True,True,2,2018-08-11T14:00:00Z,11 Aug 15:00,...,True,"[{'goals_scored': {'a': [], 'h': [{'value': 1,...",5,3,0.0,2,2,2.0,2018-19,NaN
3,987594.0,2018-08-10T18:00:00Z,10 Aug 19:00,1,2.0,True,True,3,2018-08-11T14:00:00Z,11 Aug 15:00,...,True,"[{'goals_scored': {'a': [{'value': 1, 'element...",7,2,2.0,9,2,0.0,2018-19,NaN
4,987595.0,2018-08-10T18:00:00Z,10 Aug 19:00,1,2.0,True,True,4,2018-08-11T14:00:00Z,11 Aug 15:00,...,True,"[{'goals_scored': {'a': [{'value': 1, 'element...",6,2,3.0,10,4,0.0,2018-19,NaN


Shape: (3040, 22)


## 6. Load Teams Data
Implement and run the function to load and map team names to IDs for each season.

In [15]:
def load_teams(data_root=DATA_ROOT):
    """Load and map team names to IDs."""
    teams_by_season = {}
    print("\nLoading Teams Data...")
    for season in sorted(os.listdir(data_root)):
        season_path = os.path.join(data_root, season)
        if not os.path.isdir(season_path):
            continue
        teams_file = os.path.join(season_path, "teams.csv")
        if os.path.exists(teams_file):
            try:
                teams_df = pd.read_csv(teams_file, encoding="utf-8")
                teams_df["id"] = pd.to_numeric(teams_df["id"], errors="coerce").astype("Int64")
                teams_by_season[season] = dict(zip(teams_df["name"], teams_df["id"]))
            except Exception as e:
                print(f" ERROR loading teams for {season}: {e}")
    if teams_by_season:
        print(f" → Loaded teams for {len(teams_by_season)} seasons.")
    return teams_by_season

In [16]:
# Load teams data
teams_by_season = load_teams()
print(f"Loaded teams for {len(teams_by_season)} seasons.")


Loading Teams Data...
 → Loaded teams for 7 seasons.
Loaded teams for 7 seasons.


In [17]:
# Debug: Check what seasons were actually loaded
print("\nDEBUG: Season distribution:")
print(df_gws["season"].value_counts().sort_index())
print(f"\nTotal rows per season:")
for season in sorted(df_gws["season"].unique()):
    count = len(df_gws[df_gws["season"] == season])
    print(f"  {season}: {count} rows")


DEBUG: Season distribution:
season
2018-19    21790
2019-20    16556
2020-21    24365
2021-22    25447
2022-23    26505
2023-24    29725
2024-25    27605
2025-26     7311
Name: count, dtype: int64

Total rows per season:
  2018-19: 21790 rows
  2019-20: 16556 rows
  2020-21: 24365 rows
  2021-22: 25447 rows
  2022-23: 26505 rows
  2023-24: 29725 rows
  2024-25: 27605 rows
  2025-26: 7311 rows


In [18]:
# ========================================
# CELL 2C: Direct Test (Run This First!)
# ========================================
test_names = ["aaron cresswell 376", "aaron connolly 534", "Aaron Connolly 534", "Son Heung-Min 123"]
for test in test_names:
    result = normalize_player_name(test)
    print(f"'{test}' -> '{result}'")

'aaron cresswell 376' -> 'aaron cresswell'
'aaron connolly 534' -> 'aaron connolly'
'Aaron Connolly 534' -> 'aaron connolly'
'Son Heung-Min 123' -> 'son heungmin'


## 7. Clean and Enrich Dataset
Implement and run the function to clean, rename, and enrich the gameweek data, including merging with fixtures and teams data, and adding injury/unavailable indicators.

In [19]:

def clean_dataset(df: pd.DataFrame, fixtures: pd.DataFrame, teams_by_season: dict) -> pd.DataFrame:
    """Cleans, renames, and enriches the gameweek data."""
    print("\nCleaning and Enriching Dataset...")
    print(f"  -> Initial rows: {len(df):,}")
    
    # 1. Define columns to keep and rename mapping
    keep_cols = [
    "element", "name", "web_name", "team", "opponent_team", "was_home", "season", "Gameweek",
    "minutes", "goals_scored", "assists", "clean_sheets", "goals_conceded",
    "yellow_cards", "red_cards", "bonus", "total_points",
    "influence", "creativity", "threat", "ict_index", "position"
]
    
    col_map = {
        "element": "Code", "name": "Player Name",  "web_name": "Web Name","team": "Player Team Name",
        "opponent_team": "Opponent ID", "was_home": "Is Home", "minutes": "Minutes Played",
        "goals_scored": "Goals Scored", "assists": "Assists", "clean_sheets": "Clean Sheet",
        "goals_conceded": "Goals Conceded", "yellow_cards": "Yellow Card", "red_cards": "Red Cards",
        "bonus": "Bonus Points", "total_points": "Total Points", "influence": "Influence",
        "creativity": "Creativity", "threat": "Threat", "ict_index": "ICT Index",
        "position": "Position"
    }
    
    # 2. Filter and rename columns
    available_cols = [c for c in keep_cols if c in df.columns]
    if "name" not in available_cols:
        print("  FATAL: 'name' column missing!")
        return df 
        
    df = df[available_cols].copy()
    rename_map = {k: v for k, v in col_map.items() if k in df.columns}
    df.rename(columns=rename_map, inplace=True)

    # 3. NORMALIZE PLAYER NAMES (using global function)
    if 'Player Name' in df.columns:
        print("  -> Normalizing player names...")
        df["Player Name Norm"] = df["Player Name"].apply(normalize_player_name)
        # DO NOT overwrite Player Name here - keep original for reference
        print(f"  Normalized {len(df['Player Name Norm'].unique())} unique players")
    else:
        print("  CRITICAL ERROR: 'Player Name' column missing after renaming.")
        return df
    
    # 4. Map team names to IDs
    df["Player Team ID"] = df.apply(
        lambda r: teams_by_season.get(r["season"], {}).get(r["Player Team Name"]), axis=1
    )
    
    # 5. Map opponent team names
    def get_team_name(row):
        season_teams = teams_by_season.get(row["season"], {})
        id_to_name = {v: k for k, v in season_teams.items()}
        if pd.notna(row["Opponent ID"]):
            try:
                return id_to_name.get(int(row["Opponent ID"]))
            except:
                return pd.NA
        return pd.NA

    df["Opponent Name"] = df.apply(get_team_name, axis=1)

    # 6. Convert data types
    df["Gameweek"] = pd.to_numeric(df["Gameweek"], errors="coerce").astype(pd.Int64Dtype())
    df["Player Team ID"] = pd.to_numeric(df["Player Team ID"], errors="coerce").astype(pd.Int64Dtype())
    df["Opponent ID"] = pd.to_numeric(df["Opponent ID"], errors="coerce").astype(pd.Int64Dtype())
    if "Is Home" in df.columns:
        df["Is Home"] = df["Is Home"].astype(bool)

    # 7. Map Opponent Difficulty from fixtures
    df["Opponent Difficulty"] = np.nan
    
    if not fixtures.empty and teams_by_season and "Is Home" in df.columns:
        print("  -> Mapping opponent difficulty...")
        try:
            fx = fixtures.copy()
            fx.rename(columns={
                "event": "Gameweek", "team_h": "team_h_id", "team_a": "team_a_id",
                "team_h_difficulty": "Home Difficulty", "team_a_difficulty": "Away Difficulty"
            }, inplace=True)
            fx["team_h_id"] = pd.to_numeric(fx["team_h_id"], errors="coerce").astype(pd.Int64Dtype())
            fx["team_a_id"] = pd.to_numeric(fx["team_a_id"], errors="coerce").astype(pd.Int64Dtype())
            fx["Gameweek"] = pd.to_numeric(fx["Gameweek"], errors="coerce").astype(pd.Int64Dtype())
            fx = fx[["season", "Gameweek", "team_h_id", "team_a_id", "Home Difficulty", "Away Difficulty"]].drop_duplicates()

            # Home games
            home_mask = df["Is Home"] == True
            for idx in df[home_mask].index:
                row = df.loc[idx]
                fx_match = fx[(fx["season"] == row["season"]) & 
                              (fx["Gameweek"] == row["Gameweek"]) & 
                              (fx["team_h_id"] == row["Player Team ID"])]
                if not fx_match.empty:
                    df.loc[idx, "Opponent Difficulty"] = fx_match.iloc[0]["Home Difficulty"]

            # Away games
            away_mask = df["Is Home"] == False
            for idx in df[away_mask].index:
                row = df.loc[idx]
                fx_match = fx[(fx["season"] == row["season"]) & 
                              (fx["Gameweek"] == row["Gameweek"]) & 
                              (fx["team_a_id"] == row["Player Team ID"])]
                if not fx_match.empty:
                    df.loc[idx, "Opponent Difficulty"] = fx_match.iloc[0]["Away Difficulty"]

            print(f"  Opponent Difficulty filled: {df['Opponent Difficulty'].notna().sum():,}/{len(df):,} rows")
        except Exception as e:
            print(f"  ERROR mapping difficulty: {str(e)[:100]}")
    
    # 8. Sort data
    df = df.sort_values(["Player Name Norm", "season", "Gameweek"]).reset_index(drop=True)

    # 9. Add injury flag (3+ consecutive 0-minute games)
    print("  -> Adding injury/unavailable flags...")
    df["Injury/Unavailable"] = 0
    if "Minutes Played" in df.columns:
        for player_norm in df["Player Name Norm"].unique():
            mask = df["Player Name Norm"] == player_norm
            player_data = df.loc[mask].copy()

            consecutive_zeros = 0
            injury_flags = []

            for minutes in player_data["Minutes Played"]:
                if minutes == 0:
                    consecutive_zeros += 1
                    injury_flags.append(1 if consecutive_zeros >= 3 else 0)
                else:
                    consecutive_zeros = 0
                    injury_flags.append(0)

            df.loc[mask, "Injury/Unavailable"] = injury_flags

    print(f"  Cleaning complete! Final rows: {len(df):,}")
    return df

In [20]:
# =============================================================
# 🧩 Harmonize 2025–26 merged data (no impact on other seasons)
# =============================================================

print("\n🔧 Harmonizing 2025–26 columns with legacy format...")

mask_2025 = (df_gws['season'] == '2025-26')

# 1️⃣ Ensure these columns exist (needed downstream)
for col in ['opponent_team', 'was_home']:
    if col not in df_gws.columns:
        df_gws[col] = pd.NA

# 2️⃣ Fill 'name' column ONLY for 2025–26
full_name_2025 = (
    df_gws.loc[mask_2025, 'first_name'].fillna('') + ' ' +
    df_gws.loc[mask_2025, 'second_name'].fillna('')
).str.strip().replace('', pd.NA)

df_gws.loc[mask_2025, 'name'] = (
    df_gws.loc[mask_2025, 'name']          # αν υπάρχει ήδη
    .fillna(full_name_2025)                # αλλιώς first+second
    .fillna(df_gws.loc[mask_2025, 'web_name'])  # αλλιώς web_name
)

# 3️⃣ Debug info
print("✅ Harmonization complete (2025–26 only).")
print("  Rows:", mask_2025.sum())
print("  Non-null 'name' values:", df_gws.loc[mask_2025, 'name'].notna().sum())
print("  Non-null 'web_name' values:", df_gws.loc[mask_2025, 'web_name'].notna().sum())




🔧 Harmonizing 2025–26 columns with legacy format...
✅ Harmonization complete (2025–26 only).
  Rows: 7311
  Non-null 'name' values: 7311
  Non-null 'web_name' values: 7311


In [21]:
print("\n🧪 Season presence check BEFORE cleaning:")
print(sorted(df_gws['season'].unique()))
print("Rows 2025–26:", (df_gws['season'] == '2025-26').sum())
print("Sample names 2025–26:")
display(df_gws.loc[df_gws['season'] == '2025-26', ['name', 'web_name', 'team']].head(5))



🧪 Season presence check BEFORE cleaning:
['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Rows 2025–26: 7311
Sample names 2025–26:


,name,web_name,team
171993,David Raya Martín,Raya,Arsenal
171994,Kepa Arrizabalaga Revuelta,Arrizabalaga,Arsenal
171995,Karl Hein,Hein,Arsenal
171996,Tommy Setford,Setford,Arsenal
171997,Gabriel dos Santos Magalhães,Gabriel,Arsenal


In [22]:
# Clean and enrich the dataset
df_clean = clean_dataset(df_gws, fixtures, teams_by_season)

#df_clean['Player Name'] = df_clean['Player Name Norm']

print(df_clean['season'].unique())


# Preview cleaned data
print("\nPreview of cleaned data:")
display(df_clean.head(10))
print(f"\nShape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")




Cleaning and Enriching Dataset...
  -> Initial rows: 179,304
  -> Normalizing player names...
  Normalized 2327 unique players
  -> Mapping opponent difficulty...
  Opponent Difficulty filled: 137,303/179,304 rows
  -> Adding injury/unavailable flags...
  Cleaning complete! Final rows: 179,304
['2024-25' '2025-26' '2019-20' '2020-21' '2021-22' '2023-24' '2018-19'
 '2022-23']

Preview of cleaned data:


,Code,Player Name,Web Name,Player Team Name,Opponent ID,Is Home,season,Gameweek,Minutes Played,Goals Scored,...,Influence,Creativity,Threat,ICT Index,Position,Player Name Norm,Player Team ID,Opponent Name,Opponent Difficulty,Injury/Unavailable
0,774,Aaron Anselmino,Anselmino,Chelsea,5,False,2024-25,25,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Brighton,3.0,0
1,774,Aaron Anselmino,Anselmino,Chelsea,2,False,2024-25,26,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Aston Villa,4.0,0
2,774,Aaron Anselmino,Anselmino,Chelsea,17,True,2024-25,27,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Southampton,1.0,1
3,774,Aaron Anselmino,Anselmino,Chelsea,11,True,2024-25,28,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Leicester,1.0,1
4,774,Aaron Anselmino,Anselmino,Chelsea,1,False,2024-25,29,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Arsenal,5.0,1
5,774,Aaron Anselmino,Anselmino,Chelsea,18,True,2024-25,30,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Spurs,2.0,1
6,774,Aaron Anselmino,Anselmino,Chelsea,4,False,2024-25,31,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Brentford,3.0,1
7,774,Aaron Anselmino,Anselmino,Chelsea,10,True,2024-25,32,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Ipswich,2.0,1
8,774,Aaron Anselmino,Anselmino,Chelsea,9,False,2024-25,33,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Fulham,3.0,1
9,774,Aaron Anselmino,Anselmino,Chelsea,8,True,2024-25,34,0,0,...,0.0,0.0,0.0,0.0,DEF,aaron anselmino,6,Everton,3.0,1



Shape: (179304, 27)
Columns: ['Code', 'Player Name', 'Web Name', 'Player Team Name', 'Opponent ID', 'Is Home', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Yellow Card', 'Red Cards', 'Bonus Points', 'Total Points', 'Influence', 'Creativity', 'Threat', 'ICT Index', 'Position', 'Player Name Norm', 'Player Team ID', 'Opponent Name', 'Opponent Difficulty', 'Injury/Unavailable']


In [23]:
# Check normalization worked
print("\nSample normalized names:")
sample_names = df_clean[['Player Name', 'Player Name Norm']].drop_duplicates().head(10)
display(sample_names)


Sample normalized names:


,Player Name,Player Name Norm
0,Aaron Anselmino,aaron anselmino
14,Aarón Anselmino,aaron anselmino
24,Aaron_Connolly_534,aaron connolly
50,Aaron Connolly,aaron connolly
164,Aaron_Cresswell_402,aaron cresswell
202,Aaron_Cresswell_376,aaron cresswell
231,Aaron Cresswell,aaron cresswell
421,Aaron Hickey,aaron hickey
545,Aaron_Lennon_83,aaron lennon
583,Aaron_Lennon_430,aaron lennon


## 8. Add Lagged Features
Implement and run the function to add rolling mean features (lagged features) for key metrics over the last 3 and 5 gameweeks.

In [24]:
def add_lagged_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates rolling mean features over the last 3 and 5 Gameweeks.
    Uses shift(1) to ensure we're only using past data.
    """
    print("\nAdding Lagged Features (Rolling Averages)...")

    df = df.sort_values(["Player Name Norm", "season", "Gameweek"]).reset_index(drop=True)

    rolling_metrics = [
        "Total Points", "Minutes Played", "Goals Scored", "Assists",
        "Goals Conceded", "ICT Index", "Threat", "Creativity", "Influence"
    ]

    for window in [3, 5]:
        for col in rolling_metrics:
            new_col_name = f"Avg_{col}_L{window}"
            df[new_col_name] = df.groupby(["Player Name Norm", "season"])[col] \
                .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
            df[new_col_name] = df[new_col_name].fillna(0)

    print(f"  Added {len(rolling_metrics) * 2} lagged features.")
    return df

In [25]:
# Add lagged features
df_lagged = add_lagged_features(df_clean)
# 1. Clean and enrich the dataset (Uses the fixed normalize_player_name)
#df_lagged = clean_dataset(df_gws, fixtures, teams_by_season)

# Preview with lagged features
print("\nPreview with lagged features:")
display(df_lagged.head(10))
lagged_cols = [col for col in df_lagged.columns if col.startswith("Avg_")]
print(f"\nLagged features added: {lagged_cols}")


Adding Lagged Features (Rolling Averages)...
  Added 18 lagged features.

Preview with lagged features:


,Code,Player Name,Web Name,Player Team Name,Opponent ID,Is Home,season,Gameweek,Minutes Played,Goals Scored,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
0,774,Aaron Anselmino,Anselmino,Chelsea,5,False,2024-25,25,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,774,Aaron Anselmino,Anselmino,Chelsea,2,False,2024-25,26,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,774,Aaron Anselmino,Anselmino,Chelsea,17,True,2024-25,27,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,774,Aaron Anselmino,Anselmino,Chelsea,11,True,2024-25,28,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,774,Aaron Anselmino,Anselmino,Chelsea,1,False,2024-25,29,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,774,Aaron Anselmino,Anselmino,Chelsea,18,True,2024-25,30,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,774,Aaron Anselmino,Anselmino,Chelsea,4,False,2024-25,31,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,774,Aaron Anselmino,Anselmino,Chelsea,10,True,2024-25,32,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,774,Aaron Anselmino,Anselmino,Chelsea,9,False,2024-25,33,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,774,Aaron Anselmino,Anselmino,Chelsea,8,True,2024-25,34,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Lagged features added: ['Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Goals Conceded_L3', 'Avg_ICT Index_L3', 'Avg_Threat_L3', 'Avg_Creativity_L3', 'Avg_Influence_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5', 'Avg_Goals Conceded_L5', 'Avg_ICT Index_L5', 'Avg_Threat_L5', 'Avg_Creativity_L5', 'Avg_Influence_L5']


In [26]:
# 8B. 🔄 Load and Apply Cleaned Player UUID Mapping
# Load the professor's cleaned UUID mapping and apply it to the dataset, ensuring consistency across seasons.

# %%
# --- NEW CELL: Load and Apply Cleaned Mapping ---

CLEANED_MAPPING_FILENAME = "player_uuid_mapping_cleaned.csv"
CLEANED_MAPPING_PATH = os.path.join(OUTPUT_DIR, CLEANED_MAPPING_FILENAME)

try:
    print(f"\n📥 Loading cleaned UUID mapping from: {CLEANED_MAPPING_PATH}")
    
    if not os.path.exists(CLEANED_MAPPING_PATH):
        # Αυτό είναι ένα fail-safe αν δεν βρεθεί το αρχείο, για να μην κρασάρει το notebook
        print(f"  ⚠️ CRITICAL: Cleaned mapping file not found! Please place it in {OUTPUT_DIR}/. Skipping mapping.")
        df_lagged['Player UUID'] = pd.NA
    else:
        # 1. Load the cleaned mapping file
        cleaned_mapping_df = pd.read_csv(CLEANED_MAPPING_PATH)
        
        # 2. Prepare maps (Normalized Name -> UUID, Web Name, Player Name)
        uuid_map = dict(zip(cleaned_mapping_df['Player Name Norm'], cleaned_mapping_df['Player UUID']))
        web_name_map = dict(zip(cleaned_mapping_df['Player Name Norm'], cleaned_mapping_df['Web Name']))
        player_name_map = dict(zip(cleaned_mapping_df['Player Name Norm'], cleaned_mapping_df['Player Name']))
        
        # 3. Apply maps to the main DataFrame (df_lagged)
        print("  -> Applying cleaned UUIDs...")
        df_lagged['Player UUID'] = df_lagged['Player Name Norm'].map(uuid_map)
        
        # 4. Ενημέρωση των ονομάτων με βάση το καθαρό αρχείο
        df_lagged['Web Name'] = df_lagged['Player Name Norm'].map(web_name_map).fillna(df_lagged['Web Name'])
        df_lagged['Player Name'] = df_lagged['Player Name Norm'].map(player_name_map) # Overwrite for max consistency
        
        print(f"✅ Cleaned UUIDs applied. Total UUIDs assigned: {df_lagged['Player UUID'].count():,}")

except Exception as e:
    print(f"  ERROR during cleaned mapping integration: {e}")
    df_lagged['Player UUID'] = pd.NA


📥 Loading cleaned UUID mapping from: ../output\player_uuid_mapping_cleaned.csv
  -> Applying cleaned UUIDs...
✅ Cleaned UUIDs applied. Total UUIDs assigned: 176,429


In [27]:
# 8C. Save Player UUID Mapping for Future Use (Cleaned Version)
# Saving the cleaned mapping locally for use by other scripts or modeling notebooks.

# %%
# Re-save the cleaned mapping to the standard location, overriding any old version.
if 'cleaned_mapping_df' in locals():
    # Σώζουμε το dataframe που φορτώσαμε από τον καθηγητή
    OUTPUT_MAPPING_PATH = os.path.join(OUTPUT_DIR, "player_uuid_mapping.csv") # Χρησιμοποιούμε το standard όνομα
    
    # Επιλέγουμε τις στήλες που θέλουμε να σώσουμε (τις βασικές του mapping)
    save_cols = ['Player UUID', 'Player Name', 'Web Name', 'Player Name Norm']
    
    # Χρησιμοποιούμε το καθαρό dataframe για να σώσουμε το mapping
    mapping_to_save = cleaned_mapping_df.reindex(columns=save_cols).drop_duplicates()
    
    mapping_to_save.to_csv(OUTPUT_MAPPING_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ Cleaned UUID mapping saved/overwritten to: {OUTPUT_MAPPING_PATH}")

✅ Cleaned UUID mapping saved/overwritten to: ../output\player_uuid_mapping.csv


In [28]:
# %% [markdown]
# ## 9. Save Player UUID Mapping (ensuring file is refreshed with correct data)

# This cell is now handled by the proper save_player_uuid_mapping function above
# Just verify the mapping was created correctly
print("\n✅ Verifying UUID mapping was saved...")
try:
    verification = pd.read_csv(os.path.join(OUTPUT_DIR, "player_uuid_mapping.csv"))
    print(f"  → Mapping file contains {len(verification)} players")
    print(f"  → Columns: {list(verification.columns)}")
    display(verification.head())
except FileNotFoundError:
    print("  ⚠️ ERROR: Mapping file not found! Check the save function above.")



✅ Verifying UUID mapping was saved...


  → Mapping file contains 2508 players
  → Columns: ['Player UUID', 'Player Name', 'Web Name', 'Player Name Norm']


,Player UUID,Player Name,Web Name,Player Name Norm
0,16b72858-75e4-4125-ad3b-e13f81e6d815,Aaron Anselmino,Anselmino,aaron anselmino
1,b3ce891a-840c-4259-95d5-28dd820c0551,Aaron Hickey,Hickey,aaron hickey
2,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,Aaron_Connolly_534,Connolly,aaron connolly
3,42309831-1ba1-4d68-80bb-754e846f2d73,Aaron_Cresswell,Cresswell,aaron cresswell
4,45bae922-b93e-4c3f-8428-5ef6392406fb,Aaron_Lennon,Lennon,aaron lennon


In [29]:
# Assign UUIDs to players
#df_clean, player_uuid_map = assign_player_uuids(df_clean)

# Save the mapping
#save_player_uuid_mapping(player_uuid_map)

# Debugging: Check for duplicate UUIDs on df_lagged (not df_clean!)
print("\nDEBUG: Checking for duplicate UUIDs...")
duplicates = df_lagged[df_lagged.duplicated(subset=['Player UUID', 'season', 'Gameweek'], keep=False)]
if not duplicates.empty:
    print(" → Duplicate UUIDs found:")
    display(duplicates)
else:
    print(" → No duplicate UUIDs found.")


DEBUG: Checking for duplicate UUIDs...
 → Duplicate UUIDs found:


,Code,Player Name,Web Name,Player Team Name,Opponent ID,Is Home,season,Gameweek,Minutes Played,Goals Scored,...,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5,Player UUID
107,72,Aaron_Connolly_534,Connolly,Brighton,7,True,2021-22,22,0,0,...,0.6,19.0,0.0,0.0,0.2,0.50,2.4,2.34,0.44,f6ebd126-7139-4c02-9071-fe0c85cb5e2d
108,72,Aaron_Connolly_534,Connolly,Brighton,6,True,2021-22,22,0,0,...,0.4,12.0,0.0,0.0,0.2,0.04,0.4,0.26,0.00,f6ebd126-7139-4c02-9071-fe0c85cb5e2d
110,72,Aaron_Connolly_534,Connolly,Brighton,18,False,2021-22,25,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,f6ebd126-7139-4c02-9071-fe0c85cb5e2d
111,72,Aaron_Connolly_534,Connolly,Brighton,13,False,2021-22,25,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,f6ebd126-7139-4c02-9071-fe0c85cb5e2d
115,72,Aaron_Connolly_534,Connolly,Brighton,11,True,2021-22,29,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,f6ebd126-7139-4c02-9071-fe0c85cb5e2d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179293,551,Zidane Iqbal,Iqbal,Man Utd,4,True,2022-23,29,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,c10c39c3-d1a2-4c97-9bab-869dbdc207c2
179297,551,Zidane Iqbal,Iqbal,Man Utd,2,True,2022-23,34,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,c10c39c3-d1a2-4c97-9bab-869dbdc207c2
179298,551,Zidane Iqbal,Iqbal,Man Utd,5,False,2022-23,34,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,c10c39c3-d1a2-4c97-9bab-869dbdc207c2
179301,551,Zidane Iqbal,Iqbal,Man Utd,3,False,2022-23,37,0,0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,c10c39c3-d1a2-4c97-9bab-869dbdc207c2


In [30]:

def find_duplicate_gameweeks(df: pd.DataFrame):
    """Identifies players appearing multiple times in same gameweek."""
    print("\n🔍 Finding Duplicate Entries (same player, same gameweek)...")
    
    duplicates = df.groupby(['Player UUID', 'season', 'Gameweek']).size().reset_index(name='count')
    duplicates = duplicates[duplicates['count'] > 1].sort_values('count', ascending=False)
    
    if len(duplicates) == 0:
        print(" No duplicates found!")
        return duplicates
    
    print(f" Found {len(duplicates)} duplicate entries")
    print(f" Max duplicates for one player-gameweek: {duplicates['count'].max()}\n")
    
    print(" Example duplicates:")
    for idx, row in duplicates.head(3).iterrows():
        player_uuid = row['Player UUID']
        season = row['season']
        gw = row['Gameweek']
        
        matching_rows = df[(df['Player UUID'] == player_uuid) & 
                          (df['season'] == season) & 
                          (df['Gameweek'] == gw)]
        
        print(f"\n    Player: {matching_rows['Player Name'].iloc[0]}")
        print(f"    Season {season}, Gameweek {gw} ({row['count']} entries)")
        print(f"    Minutes: {matching_rows['Minutes Played'].values}")
        print(f"    Points: {matching_rows['Total Points'].values}")
    
    return duplicates


def merge_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Merges duplicate player-gameweek entries using smart aggregation."""
    print("\n🔄 Merging Duplicate Rows...")
    
    initial_rows = len(df)
    
    # Define aggregation strategy
    agg_dict = {'Player Name': 'first'}
    
    for col in df.columns:
        if col in ['Player UUID', 'Player Name Norm', 'season', 'Gameweek', 'Player Name']:
            continue
        elif col == 'Minutes Played':
            agg_dict[col] = 'max'  # Take highest minutes
        elif col in ['Total Points', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 
                      'Yellow Card', 'Red Cards', 'Bonus Points', 'Threat', 'Influence', 
                      'Creativity', 'ICT Index']:
            agg_dict[col] = 'sum'  # Sum performance metrics
        elif col in ['Is Home', 'Injury/Unavailable']:
            agg_dict[col] = 'max'
        elif col.startswith('Avg_'):
            agg_dict[col] = 'mean'  # Average the averages
        else:
            agg_dict[col] = 'first'  # Keep first for text columns
            
    df_merged = df.groupby(['Player UUID', 'Player Name Norm', 'season', 'Gameweek'], 
                           as_index=False).agg(agg_dict)
    
    final_rows = len(df_merged)
    merged_count = initial_rows - final_rows
    
    print(f"  → Rows before merge: {initial_rows:,}")
    print(f"  → Rows after merge: {final_rows:,}")
    print(f"  Rows merged: {merged_count:,}")
    
    return df_merged


def verify_no_duplicates(df: pd.DataFrame) -> bool:
    """Verifies no duplicates remain."""
    print("\n✓ Verifying No Duplicates Remain...")
    
    max_dupes = df.groupby(['Player UUID', 'season', 'Gameweek']).size().max()
    
    if max_dupes == 1:
        print(f" Success! Each player appears exactly once per gameweek.")
        return True
    else:
        print(f"  ⚠️  Warning: Maximum duplicates still: {max_dupes}")
        return False



In [31]:
# NEW: Check for duplicate entries per gameweek
duplicates = find_duplicate_gameweeks(df_lagged)

# Right before: df_lagged = merge_duplicate_rows(df_lagged)
# Add this line:
#df_lagged['Player Name'] = df_lagged['Player Name Norm']

# Merge the duplicates
#df_lagged = merge_duplicate_rows(df_lagged)

# Merge Duplicates (if any exist)
if len(duplicates) > 0:
    df_lagged = merge_duplicate_rows(df_lagged)
else:
    print("\n✅ No duplicates to merge!")
    
# Verify they're gone
verify_no_duplicates(df_lagged)



🔍 Finding Duplicate Entries (same player, same gameweek)...
 Found 7401 duplicate entries
 Max duplicates for one player-gameweek: 4

 Example duplicates:

    Player: Ben_Davies
    Season 2021-22, Gameweek 29 (4 entries)
    Minutes: [0 0 86 90]
    Points: [0 0 1 6]

    Player: Ben_Davies
    Season 2021-22, Gameweek 36 (4 entries)
    Minutes: [0 0 90 81]
    Points: [0 0 1 5]

    Player: Ben_Davies
    Season 2020-21, Gameweek 26 (4 entries)
    Minutes: [0 0 0 90]
    Points: [0 0 0 6]

🔄 Merging Duplicate Rows...
  → Rows before merge: 179,304
  → Rows after merge: 168,972
  Rows merged: 10,332

✓ Verifying No Duplicates Remain...
 Success! Each player appears exactly once per gameweek.


True

In [32]:
# 8D. Final Consistency Check (Debugging)
# Re-running the duplicate check to ensure the merging process was successful and no inconsistencies remain.

# %%
print("\n🔎 FINAL CONSISTENCY CHECK AFTER MERGING...")

# 1. Έλεγχος για διπλότυπα Player UUID + season + Gameweek
max_dupes = df_lagged.groupby(['Player UUID', 'season', 'Gameweek']).size().max()

if max_dupes == 1:
    print("✅ Success! The data is now fully consistent (max entries per player-gameweek: 1).")
elif max_dupes is np.nan:
    # Μπορεί να συμβεί αν το df είναι άδειο
    print("⚠️ Warning: DataFrame is empty or UUIDs are missing.")
else:
    print(f"❌ CRITICAL ERROR: Inconsistencies remain! Max duplicates: {max_dupes}")
    print("Sample inconsistent rows:")
    
    # Εμφάνιση των γραμμών που ακόμα έχουν πρόβλημα
    inconsistent_groups = df_lagged.groupby(['Player UUID', 'season', 'Gameweek']).filter(lambda x: len(x) > 1)
    
    if not inconsistent_groups.empty:
        # Παίρνουμε ένα μικρό δείγμα (π.χ. 5 διπλότυπες ομάδες)
        sample_group_keys = inconsistent_groups[['Player UUID', 'season', 'Gameweek']].drop_duplicates().head(5)
        
        for index, row in sample_group_keys.iterrows():
            player_uuid = row['Player UUID']
            season = row['season']
            gw = row['Gameweek']
            
            error_rows = df_lagged[(df_lagged['Player UUID'] == player_uuid) & 
                                   (df_lagged['season'] == season) & 
                                   (df_lagged['Gameweek'] == gw)]
            
            print(f"\n    Player: {error_rows['Player Name'].iloc[0]} (UUID: {player_uuid[:8]}...)")
            print(f"    Season {season}, Gameweek {gw} ({len(error_rows)} entries)")
            display(error_rows[['Player Team Name', 'Minutes Played', 'Total Points', 'Opponent Name']].head())


🔎 FINAL CONSISTENCY CHECK AFTER MERGING...
✅ Success! The data is now fully consistent (max entries per player-gameweek: 1).


## 9. Save Final Dataset to CSV
Select final columns, sort the DataFrame, and save the processed data to a CSV file.

In [33]:
final_cols = [
    "Player UUID", "Code", "Player Name", "Web Name", "Player Team Name", "season", "Gameweek",
    "Minutes Played", "Goals Scored", "Assists", "Clean Sheet", "Goals Conceded",
    "Yellow Card", "Red Cards", "Bonus Points", "Total Points", "Threat",
    "ICT Index", "Influence", "Creativity", "Opponent Name", "Opponent Difficulty",
    "Is Home", "Position", "Injury/Unavailable"
]

lagged_cols = [col for col in df_lagged.columns if col.startswith("Avg_")]
final_cols.extend(lagged_cols)

df_final = df_lagged.reindex(columns=final_cols, fill_value=np.nan)
df_final = df_final.sort_values(by=["Player Name", "season", "Gameweek"], ignore_index=True)
df_final.insert(0, "A", df_final.index)

print(f" Final dataset prepared")
print(f" Shape: {df_final.shape}")

 Final dataset prepared
 Shape: (168972, 44)


In [34]:
# 9A. Backup Current training_data.csv
# Saving a timestamped backup of the existing training_data.csv before writing the new, cleaned version.

# %%
import datetime
import shutil

# Ορίζουμε το όνομα του αρχείου backup
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_FILE = os.path.join(OUTPUT_DIR, f"training_data_backup_{timestamp}.csv")

# Ελέγχουμε αν υπάρχει ήδη ένα training_data.csv για να το κάνουμε backup
if os.path.exists(OUTPUT_FILE):
    print(f"\n💾 Creating backup of existing file to: {BACKUP_FILE}")
    try:
        shutil.copyfile(OUTPUT_FILE, BACKUP_FILE)
        print("✅ Backup successful.")
    except Exception as e:
        print(f"⚠️ ERROR during backup: {e}")
else:
    print("\n⏩ No existing training_data.csv found to backup.")


💾 Creating backup of existing file to: ../output\training_data_backup_20251112_143129.csv
✅ Backup successful.


In [35]:
# Save final dataset
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"🎉 PIPELINE COMPLETE! 🎉")
print(f"{'='*60}")
print(f"  💾 Output saved to: {OUTPUT_FILE}")
print(f"  📊 Total rows: {len(df_final):,}")
print(f"  📋 Total features: {len(df_final.columns) - 1}")
print(f"  🆔 Unique players: {df_final['Player UUID'].nunique():,}")
print(f"{'='*60}\n")


🎉 PIPELINE COMPLETE! 🎉
  💾 Output saved to: ../output/training_data.csv
  📊 Total rows: 168,972
  📋 Total features: 43
  🆔 Unique players: 2,098



In [36]:
# Final dataset preview and summary
print("Final Dataset Preview:")
display(df_final.head(20))

print("\n Summary Statistics:")
print(f"  Seasons: {sorted(df_final['season'].unique())}")
print(f"  Gameweeks per season: {df_final.groupby('season')['Gameweek'].max().to_dict()}")
print(f"  Total player-gameweek records: {len(df_final):,}")

Final Dataset Preview:


,A,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
0,0,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,5,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,30,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,6,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,31,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,7,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,32,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,8,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,33,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,9,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,34,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



 Summary Statistics:
  Seasons: ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  Gameweeks per season: {'2018-19': 38, '2019-20': 29, '2020-21': 38, '2021-22': 38, '2022-23': 38, '2023-24': 38, '2024-25': 38, '2025-26': 10}
  Total player-gameweek records: 168,972


In [37]:
# Verify UUID mapping
print("\n🔍 Verifying UUID Mapping:")
sample_players = df_final['Player Name'].value_counts().head(5)
print("\nTop 5 players by appearances:")
for player_name in sample_players.index:
    player_data = df_final[df_final['Player Name'] == player_name]
    uuid = player_data['Player UUID'].iloc[0]
    appearances = len(player_data)
    print(f"  {player_name}: {appearances} games | UUID: {uuid}")

# Check if any player has multiple UUIDs (shouldn't happen!)
print("\n🔍 Checking for players with multiple UUIDs...")
player_uuid_check = df_final.groupby('Player Name')['Player UUID'].nunique()
multi_uuid_players = player_uuid_check[player_uuid_check > 1]
if len(multi_uuid_players) == 0:
    print("  ✅ Perfect! Each player has exactly one UUID")
else:
    print(f"  ⚠️  WARNING: {len(multi_uuid_players)} players have multiple UUIDs!")
    display(multi_uuid_players)


🔍 Verifying UUID Mapping:

Top 5 players by appearances:
  Issa_Diop_409: 262 games | UUID: b7504e7c-8346-4e73-8614-f7c913d3a573
  Ben_Davies: 261 games | UUID: 3e20649c-3f65-4d75-998a-a137be556e95
  Declan_Rice: 260 games | UUID: 24a4a261-c963-4dfd-90e6-3f8a4be8e872
  Morgan_Gibbs-White_448: 260 games | UUID: af69bd10-f1c9-4233-b5c2-97640dfd3165
  Willy_Boly_423: 260 games | UUID: da72fe37-7cb6-422b-b76d-3e79f884a6f3

🔍 Checking for players with multiple UUIDs...
  ✅ Perfect! Each player has exactly one UUID


In [38]:
print(f" → Unique seasons: {df_final['season'].unique()}")


 → Unique seasons: ['2024-25' '2025-26' '2022-23' '2023-24' '2019-20' '2020-21' '2021-22'
 '2018-19']


In [39]:
# See if any player appears multiple times in the same gameweek
df_final.groupby(['Player Name', 'season', 'Gameweek']).size().max()



np.int64(1)

In [40]:
# Sanity check before saving
#print("Rows:", len(player_uuid_map_df))
#print("Unique UUIDs:", player_uuid_map_df['Player UUID'].nunique())
#player_uuid_map_df.head()

# Save
#player_uuid_map_df.to_csv("data/player_maps/player_uuid_mapping.csv", index=False)
#print("✅ Saved updated mapping file.")


In [41]:
[x for x in globals().keys() if "uuid" in x.lower()]


['uuid_module', 'uuid_map', 'uuid', 'player_uuid_check', 'multi_uuid_players']